# Hippo Encoder Colab

Train the text distillation model, run embedding evaluation, and test the sparse two-array region benchmark.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/CameronBadman/Hippo-encoder.git'
WORKDIR = Path('/content/Hippo-encoder')

if WORKDIR.exists():
    !rm -rf /content/Hippo-encoder

!git clone {REPO_URL} {WORKDIR}
%cd /content/Hippo-encoder


In [ ]:
%cd /content/Hippo-encoder
!python -m pip install --upgrade pip
!pip install -e .


In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/hippo_encoder_data')
RUN_ROOT = Path('/content/drive/MyDrive/hippo_encoder_runs')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
RUN_DIR = RUN_ROOT / 'distill-bge-small'

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print('TRAIN_JSONL =', TRAIN_JSONL)
print('RUN_DIR =', RUN_DIR)


In [ ]:
from pathlib import Path
import json

config = {
    'teacher_model_name': 'intfloat/e5-base-v2',
    'student_model_name': 'BAAI/bge-small-en-v1.5',
    'dataset_jsonl': '/content/drive/MyDrive/hippo_encoder_data/train.jsonl',
    'output_dir': '/content/drive/MyDrive/hippo_encoder_runs/distill-bge-small',
    'max_text_length': 64,
    'batch_size': 32,
    'num_epochs': 3,
    'learning_rate': 1e-4,
    'weight_decay': 1e-2,
    'log_every': 20,
    'save_every': 100000,
    'num_workers': 2,
    'teacher_text_weight': 1.0,
    'hidden_state_weight': 0.2,
    'contrastive_weight': 0.0,
    'normalize_targets': True,
    'gradient_clip_norm': 1.0,
    'warmup_steps': 100,
    'seed': 42,
}

runtime_config_path = Path('/content/Hippo-encoder/configs/colab_run.json')
runtime_config_path.write_text(json.dumps(config, indent=2), encoding='utf-8')
print(runtime_config_path.read_text())


In [ ]:
%cd /content/Hippo-encoder
!python scripts/prepare_text_dataset.py \
  --dataset ag_news \
  --split train \
  --text-column text \
  --limit 20000 \
  --prefix 'passage: ' \
  --output /content/drive/MyDrive/hippo_encoder_data/train.jsonl


In [ ]:
%cd /content/Hippo-encoder
!python -m hippo_encoder.train --config configs/colab_run.json


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

teacher_name = 'intfloat/e5-base-v2'
student_ckpt = '/content/drive/MyDrive/hippo_encoder_runs/distill-bge-small/epoch-2'

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_name)
teacher_model = AutoModel.from_pretrained(teacher_name).to(device).eval()

student_tokenizer = AutoTokenizer.from_pretrained(f'{student_ckpt}/tokenizer')
student_backbone = AutoModel.from_pretrained(f'{student_ckpt}/backbone').to(device).eval()

heads = torch.load(f'{student_ckpt}/heads.pt', map_location=device)
embed_head = torch.nn.Linear(student_backbone.config.hidden_size, teacher_model.config.hidden_size).to(device)
embed_head.load_state_dict(heads['embed_head'])
embed_head.eval()

def masked_mean(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    masked = hidden_states * mask
    denom = mask.sum(dim=1).clamp(min=1.0)
    return masked.sum(dim=1) / denom

@torch.no_grad()
def teacher_embed(texts):
    batch = teacher_tokenizer(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
    batch = {k: v.to(device) for k, v in batch.items()}
    out = teacher_model(**batch, return_dict=True)
    emb = masked_mean(out.last_hidden_state, batch['attention_mask'])
    return F.normalize(emb, dim=-1)

@torch.no_grad()
def student_embed(texts):
    batch = student_tokenizer(texts, padding=True, truncation=True, max_length=64, return_tensors='pt')
    batch = {k: v.to(device) for k, v in batch.items()}
    out = student_backbone(**batch, return_dict=True)
    pooled = masked_mean(out.last_hidden_state, batch['attention_mask'])
    emb = embed_head(pooled)
    return F.normalize(emb, dim=-1)

eval_texts = [
    'query: red chair near the window',
    'query: a scarlet chair beside the window',
    'query: blue mug on the desk',
    'query: coffee cup sitting on the office desk',
    'query: dog running through the grass',
    'query: a small dog sprinting across a field',
    'query: stock market falls after weak earnings',
    'query: company shares drop after poor results',
    'query: basketball team wins championship game',
    'query: the team secured the title in the final match',
    'query: person standing beside a parked bicycle',
    'query: someone next to a bike on the street',
]

t = teacher_embed(eval_texts)
s = student_embed(eval_texts)
pair_cos = F.cosine_similarity(t, s, dim=-1)
print('Mean teacher-student cosine:', pair_cos.mean().item())


In [ ]:
%cd /content/Hippo-encoder
!git pull --ff-only
!python scripts/benchmark_region_membership.py \
  --cases benchmarks/sample_region_cases.json \
  --student-checkpoint /content/drive/MyDrive/hippo_encoder_runs/distill-bge-small/epoch-2 \
  --inside-threshold 0.75 \
  --radius-scale 1.5


## Notes

- This notebook is text-only. The dataset rows look like `{"text": "passage: ..."}`.
- The sparse region benchmark uses two hydrated arrays around the anchor embedding: `minus` and `plus`.
- `inside-threshold` and `radius-scale` are benchmark calibration knobs, not the final learned region program.
- If you change `output_dir`, update the checkpoint path in the eval and region benchmark cells.